In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
import warnings
import joblib
warnings.filterwarnings('ignore')

print("✅ Libraries imported")

✅ Libraries imported


In [2]:
# Load the data we saved during exploration
X_train = pd.read_csv('../data/processed/UNSW_X_train.csv')
X_val = pd.read_csv('../data/processed/UNSW_X_val.csv')
X_test = pd.read_csv('../data/processed/UNSW_X_test.csv')

y_train = pd.read_csv('../data/processed/UNSW_y_train.csv').values.ravel()
y_val = pd.read_csv('../data/processed/UNSW_y_val.csv').values.ravel()
y_test = pd.read_csv('../data/processed/UNSW_y_test.csv').values.ravel()

print("="*60)
print("DATA LOADED SUCCESSFULLY")
print("="*60)
print(f"Training set:   {X_train.shape} with {len(y_train)} labels")
print(f"Validation set: {X_val.shape} with {len(y_val)} labels")
print(f"Test set:       {X_test.shape} with {len(y_test)} labels")

DATA LOADED SUCCESSFULLY
Training set:   (122738, 34) with 122738 labels
Validation set: (52603, 34) with 52603 labels
Test set:       (82332, 34) with 82332 labels


In [3]:
print("="*60)
print("IDENTIFYING FEATURE TYPES")
print("="*60)

# Based on our exploration, we know the categorical columns
categorical_cols = ['proto', 'service', 'state']
numerical_cols = [col for col in X_train.columns if col not in categorical_cols]

print(f"Categorical features ({len(categorical_cols)}):")
for col in categorical_cols:
    print(f"  - {col} (unique values: {X_train[col].nunique()})")

print(f"\nNumerical features ({len(numerical_cols)}):")
print(f"  First 10: {numerical_cols[:10]}")
print(f"  Total: {len(numerical_cols)} numerical features")

IDENTIFYING FEATURE TYPES
Categorical features (3):
  - proto (unique values: 133)
  - service (unique values: 13)
  - state (unique values: 7)

Numerical features (31):
  First 10: ['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'dload', 'sloss', 'dloss']
  Total: 31 numerical features


In [4]:
print("="*60)
print("  CLASS IMBALANCE CHECK")
print("="*60)

train_normal = sum(y_train == 0)
train_attack = sum(y_train == 1)
val_normal = sum(y_val == 0)
val_attack = sum(y_val == 1)

print("TRAINING SET:")
print(f"  Normal: {train_normal} ({train_normal/len(y_train)*100:.2f}%)")
print(f"  Attack: {train_attack} ({train_attack/len(y_train)*100:.2f}%)")

print("\nVALIDATION SET:")
print(f"  Normal: {val_normal} ({val_normal/len(y_val)*100:.2f}%)")
print(f"  Attack: {val_attack} ({val_attack/len(y_val)*100:.2f}%)")

# Calculate class weights for handling imbalance
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print("\n RECOMMENDED CLASS WEIGHTS (for SVM/CNN):")
print(f"  Normal (0): {class_weight_dict[0]:.4f}")
print(f"  Attack (1): {class_weight_dict[1]:.4f}")
print("\n  These weights give more importance to the minority class (Normal)")

  CLASS IMBALANCE CHECK
TRAINING SET:
  Normal: 39200 (31.94%)
  Attack: 83538 (68.06%)

VALIDATION SET:
  Normal: 16800 (31.94%)
  Attack: 35803 (68.06%)

 RECOMMENDED CLASS WEIGHTS (for SVM/CNN):
  Normal (0): 1.5655
  Attack (1): 0.7346

  These weights give more importance to the minority class (Normal)


In [5]:
print("="*60)
print("ENCODING CATEGORICAL FEATURES")
print("="*60)

# Create dictionary to store encoders
encoders = {}
X_train_encoded = X_train.copy()
X_val_encoded = X_val.copy()
X_test_encoded = X_test.copy()

# Encode each categorical column
for col in categorical_cols:
    le = LabelEncoder()
    
    # Fit on training data only
    X_train_encoded[col] = le.fit_transform(X_train[col].astype(str))
    
    # Transform validation and test
    X_val_encoded[col] = X_val[col].astype(str).map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    X_test_encoded[col] = X_test[col].astype(str).map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    
    encoders[col] = le
    print(f"Encoded '{col}': {len(le.classes_)} unique values")

print(f"\nTraining shape after encoding: {X_train_encoded.shape}")

ENCODING CATEGORICAL FEATURES
Encoded 'proto': 133 unique values
Encoded 'service': 13 unique values
Encoded 'state': 7 unique values

Training shape after encoding: (122738, 34)


In [7]:
print("="*60)
print("VERIFYING ENCODING")
print("="*60)

for col in categorical_cols:
    print(f"\n{col}:")
    print(f"  Original (first 5): {X_train[col].head().tolist()}")
    print(f"  Encoded (first 5):  {X_train_encoded[col].head().tolist()}")
    print(f"  Unique values: {X_train_encoded[col].nunique()}")

VERIFYING ENCODING

proto:
  Original (first 5): ['udp', 'tcp', 'tcp', 'udp', 'unas']
  Encoded (first 5):  [119, 113, 113, 119, 120]
  Unique values: 133

service:
  Original (first 5): ['dns', '-', '-', 'dns', '-']
  Encoded (first 5):  [2, 0, 0, 2, 0]
  Unique values: 13

state:
  Original (first 5): ['CON', 'FIN', 'FIN', 'INT', 'INT']
  Encoded (first 5):  [0, 2, 2, 3, 3]
  Unique values: 7


In [8]:
print("="*60)
print("SCALING NUMERICAL FEATURES")
print("="*60)

# Initialize scaler
scaler = StandardScaler()  # Good for SVM and CNN

# Fit on training data only
X_train_scaled = X_train_encoded.copy()
X_val_scaled = X_val_encoded.copy()
X_test_scaled = X_test_encoded.copy()

# Scale numerical features
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train_encoded[numerical_cols])
X_val_scaled[numerical_cols] = scaler.transform(X_val_encoded[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test_encoded[numerical_cols])

print(f"Scaling complete!")
print(f"\nBefore scaling - 'dur' feature:")
print(f"  Min: {X_train_encoded['dur'].min():.4f}")
print(f"  Max: {X_train_encoded['dur'].max():.4f}")
print(f"  Mean: {X_train_encoded['dur'].mean():.4f}")

print(f"\nAfter scaling - 'dur' feature:")
print(f"  Min: {X_train_scaled['dur'].min():.4f}")
print(f"  Max: {X_train_scaled['dur'].max():.4f}")
print(f"  Mean: {X_train_scaled['dur'].mean():.4f}")

SCALING NUMERICAL FEATURES
Scaling complete!

Before scaling - 'dur' feature:
  Min: 0.0000
  Max: 60.0000
  Mean: 1.3681

After scaling - 'dur' feature:
  Min: -0.2100
  Max: 9.0017
  Mean: -0.0000


In [9]:
print("="*60)
print("PREPROCESSING COMPLETE!")
print("="*60)

print(f"Final training data shape: {X_train_scaled.shape}")
print(f"Final validation data shape: {X_val_scaled.shape}")
print(f"Final test data shape: {X_test_scaled.shape}")

print("\n Data is now ready for machine learning!")
print("  ✓ Categorical features encoded")
print("  ✓ Numerical features scaled")
print("  ✓ Class weights calculated for imbalance")

print("\nFirst 5 rows of preprocessed training data:")
X_train_scaled.head()

PREPROCESSING COMPLETE!
Final training data shape: (122738, 34)
Final validation data shape: (52603, 34)
Final test data shape: (82332, 34)

 Data is now ready for machine learning!
  ✓ Categorical features encoded
  ✓ Numerical features scaled
  ✓ Class weights calculated for imbalance

First 5 rows of preprocessed training data:


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,smean,dmean,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports
0,-0.209875,119,2,0,-0.133387,-0.152155,-0.049642,-0.102128,-0.560983,-0.386784,...,-0.350189,-0.167125,-0.119176,-0.041936,-0.422359,-0.555737,-0.119009,-0.119009,-0.186567,-0.127059
1,-0.171936,113,0,2,-0.016696,-0.116629,0.025606,-0.101406,-0.576709,-0.387121,...,2.949467,-0.306218,-0.119176,-0.041936,-0.546392,-0.555737,-0.119009,-0.119009,-0.186567,-0.127059
2,-0.205255,113,0,2,0.741793,0.949158,-0.005701,-0.000153,-0.529576,-0.378783,...,-0.355091,-0.016442,-0.119176,-0.041936,-0.546392,-0.555737,-0.119009,-0.119009,-0.186567,-0.127059
3,-0.210044,119,2,3,-0.133387,-0.169918,-0.049734,-0.103232,0.092201,-0.122357,...,-0.389412,-0.480084,-0.119176,-0.041936,3.422683,2.027669,-0.119009,-0.119009,-0.186567,-0.127059
4,-0.210044,120,0,3,-0.133387,-0.169918,-0.049242,-0.103232,0.092201,0.078985,...,-0.178587,-0.480084,-0.119176,-0.041936,-0.174292,-0.039056,-0.119009,-0.119009,-0.186567,-0.127059


In [11]:
# ============================================================
# CREATE MODELS/UNSW FOLDER
# ============================================================
import os

# Create models/UNSW folder if it doesn't exist
models_unsw_path = '../models/UNSW'
if not os.path.exists(models_unsw_path):
    os.makedirs(models_unsw_path)
    print(f" Created folder: {models_unsw_path}")
else:
    print(f" Folder already exists: {models_unsw_path}")

# Also ensure data/processed exists
processed_path = '../data/processed'
if not os.path.exists(processed_path):
    os.makedirs(processed_path)
    print(f" Created folder: {processed_path}")

 Created folder: ../models/UNSW


In [12]:
print("="*60)
print("SAVING PREPROCESSED DATA")
print("="*60)

# Save processed data
X_train_scaled.to_csv('../data/processed/UNSW_X_train_preprocessed.csv', index=False)
X_val_scaled.to_csv('../data/processed/UNSW_X_val_preprocessed.csv', index=False)
X_test_scaled.to_csv('../data/processed/UNSW_X_test_preprocessed.csv', index=False)

# Save labels
pd.DataFrame(y_train).to_csv('../data/processed/UNSW_y_train.csv', index=False)
pd.DataFrame(y_val).to_csv('../data/processed/UNSW_y_val.csv', index=False)
pd.DataFrame(y_test).to_csv('../data/processed/UNSW_y_test.csv', index=False)

# Save encoders and scaler for later use
joblib.dump(encoders, '../models/UNSW/encoders.pkl')
joblib.dump(scaler, '../models/UNSW/scaler.pkl')

# Save class weights for model training
joblib.dump(class_weight_dict, '../models/UNSW/class_weights.pkl')

print("Training data saved: data/processed/UNSW_X_train_preprocessed.csv")
print("Validation data saved: data/processed/UNSW_X_val_preprocessed.csv")
print("Test data saved: data/processed/UNSW_X_test_preprocessed.csv")
print("Encoders saved: models/UNSW/encoders.pkl")
print("Scaler saved: models/UNSW/scaler.pkl")
print("Class weights saved: models/UNSW/class_weights.pkl")

SAVING PREPROCESSED DATA
Training data saved: data/processed/UNSW_X_train_preprocessed.csv
Validation data saved: data/processed/UNSW_X_val_preprocessed.csv
Test data saved: data/processed/UNSW_X_test_preprocessed.csv
Encoders saved: models/UNSW/encoders.pkl
Scaler saved: models/UNSW/scaler.pkl
Class weights saved: models/UNSW/class_weights.pkl


In [13]:
print("="*60)
print("PROCESSED DATA STATISTICS")
print("="*60)

print("Training set statistics (first 5 numerical features):")
X_train_scaled[numerical_cols[:5]].describe()

print("\nCheck for any remaining issues:")
print(f"  Any NaN values? {X_train_scaled.isnull().any().any()}")
print(f"  Any infinite values? {np.isinf(X_train_scaled.select_dtypes(include=[np.number])).any().any()}")

PROCESSED DATA STATISTICS
Training set statistics (first 5 numerical features):

Check for any remaining issues:
  Any NaN values? False
  Any infinite values? False


In [14]:
# ============================================================
# VERIFY ALL SAVED FILES
# ============================================================
import os
import pandas as pd
import joblib

print("="*60)
print("VERIFYING SAVED FILES")
print("="*60)

# Check data/processed folder
print("\n DATA/PROCESSED FOLDER:")
processed_path = '../data/processed'
if os.path.exists(processed_path):
    processed_files = os.listdir(processed_path)
    unsw_files = [f for f in processed_files if 'UNSW' in f]
    
    for file in sorted(unsw_files):
        file_path = os.path.join(processed_path, file)
        size_kb = os.path.getsize(file_path) / 1024
        print(f"   {file} ({size_kb:.1f} KB)")
        
        # Quick peek at first few files
        if file.endswith('.csv'):
            try:
                df = pd.read_csv(file_path, nrows=2)
                print(f"     Shape: {df.shape}, Columns: {list(df.columns[:3])}...")
            except:
                print(f"       Could not read file")
else:
    print("   Folder not found!")

# Check models/UNSW folder
print("\n MODELS/UNSW FOLDER:")
models_path = '../models/UNSW'
if os.path.exists(models_path):
    model_files = os.listdir(models_path)
    
    for file in sorted(model_files):
        file_path = os.path.join(models_path, file)
        size_kb = os.path.getsize(file_path) / 1024
        print(f"   {file} ({size_kb:.1f} KB)")
        
        # Try to load pickle files
        if file.endswith('.pkl'):
            try:
                data = joblib.load(file_path)
                if file == 'encoders.pkl':
                    print(f"     Encoders for: {list(data.keys())}")
                elif file == 'scaler.pkl':
                    print(f"     Scaler type: {type(data).__name__}")
                elif file == 'class_weights.pkl':
                    print(f"     Class weights: {data}")
            except:
                print(f"       Could not load pickle file")
else:
    print("   Folder not found!")

print("\n" + "="*60)
print("VERIFICATION COMPLETE")
print("="*60)

VERIFYING SAVED FILES

 DATA/PROCESSED FOLDER:
   UNSW_X_test.csv (12265.4 KB)
     Shape: (2, 34), Columns: ['dur', 'proto', 'service']...
   UNSW_X_test_preprocessed.csv (51698.6 KB)
     Shape: (2, 34), Columns: ['dur', 'proto', 'service']...
   UNSW_X_train.csv (17966.6 KB)
     Shape: (2, 34), Columns: ['dur', 'proto', 'service']...
   UNSW_X_train_preprocessed.csv (77044.5 KB)
     Shape: (2, 34), Columns: ['dur', 'proto', 'service']...
   UNSW_X_val.csv (7712.3 KB)
     Shape: (2, 34), Columns: ['dur', 'proto', 'service']...
   UNSW_X_val_preprocessed.csv (33020.9 KB)
     Shape: (2, 34), Columns: ['dur', 'proto', 'service']...
   UNSW_y_test.csv (241.2 KB)
     Shape: (2, 1), Columns: ['0']...
   UNSW_y_train.csv (359.6 KB)
     Shape: (2, 1), Columns: ['0']...
   UNSW_y_val.csv (154.1 KB)
     Shape: (2, 1), Columns: ['0']...

 MODELS/UNSW FOLDER:
   class_weights.pkl (0.1 KB)
     Class weights: {0: np.float64(1.5655357142857143), 1: np.float64(0.7346237640355288)}
   encoder